# Distal-peaks pycisTopic — UMAP (Step 5)

Load assembled Mallet LDA models, select one, run clustering + UMAP, save a CellType-colored UMAP as PDF.

Run **after** the Mallet bash scripts (02 -> 03 -> 04).

## Imports + CONFIG

In [ ]:
import os
import pickle
import warnings
warnings.simplefilter(action="ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patheffects import withStroke

import pycisTopic
print("pycisTopic version:", pycisTopic.__version__)
from pycisTopic.lda_models import evaluate_models
from pycisTopic.clust_vis import find_clusters, run_umap, plot_metadata

WORK_DIR   = "/work/Home/home_py/local_data/proj/Dev_Multiome/04.regulome/02.Revision1_20260609/"
RES_DIR    = os.path.join(WORK_DIR, "results")
RUN_NAME   = "01.pycisTopic_distal_UMAP"
Mallet_DIR = os.path.join(RES_DIR, RUN_NAME, "Mallet_res")

CISTOPIC_PKL = os.path.join(RES_DIR, RUN_NAME, "cistopic_obj.pkl")
# MODEL_PREFIX = os.path.join(Mallet_DIR, RUN_NAME)   # <prefix>.<n>_topics.model.pkl

N_TOPICS_LIST = [15, 20, 25, 30, 35, 40, 45]   # must match Mallet output
SELECT_MODEL  = 35                              # number of topics to keep
SAMPLE_TAG    = "FL_distal"
OUT_PDF       = os.path.join(WORK_DIR, f"CellType_umap_{SAMPLE_TAG}.pdf")

os.chdir(WORK_DIR)


## Cell-type color dictionary

In [ ]:
celltype_colors = {
    'HSC': '#E41A1C', 'GP': '#E0FFFF', 'Granulocyte': '#B3CDE3',
    'MEMP-t': '#E6AB02', 'MEMP': '#FF7F00', 'MEP': '#CD661D',
    'MEMP-Mast-Ery': '#FDCDAC', 'MEMP-Ery': '#E9967A', 'Early-Ery': '#CD5555',
    'Late-Ery': '#8B0000', 'MEMP-MK': '#663C1F', 'MK': '#40E0D0',
    'MastP-t': '#1E90FF', 'MastP': '#1F78B4', 'Mast': '#253494',
    'MDP': '#E6F5C9', 'Monocyte': '#005A32', 'Kupffer': '#00EE00',
    'cDC1': '#B3DE69', 'cDC2': '#ADFF2F', 'pDC': '#4DAF4A', 'ASDC': '#CDC673',
    'LMP': '#FFF2AE', 'LP': '#FFD92F', 'Cycling-LP': '#FFFF33',
    'PreProB': '#FFF0F5', 'ProB-1': '#FFB5C5', 'ProB-2': '#E78AC3',
    'Large-PreB': '#CD1076', 'Small-PreB': '#FF3E96', 'IM-B': '#FF00FF',
    'NK': '#A020F0', 'ILCP': '#49006A', 'T': '#984EA3',
    'Hepatocyte': '#666666', 'Endothelia': '#000000',
}


## 1. Load cisTopic object + models

In [ ]:
with open(CISTOPIC_PKL, "rb") as fh:
    cistopic_obj = pickle.load(fh)

models = []
for n in N_TOPICS_LIST:
    with open(f"{MODEL_PREFIX}.{n}_topics.model.pkl", "rb") as fh:
        models.append(pickle.load(fh))
print(f"loaded {len(models)} models: {N_TOPICS_LIST}")


## 2. Select model and attach

In [ ]:
model = evaluate_models(models, select_model=SELECT_MODEL, return_model=True)
cistopic_obj.add_LDA_model(model)

with open(os.path.join(Mallet_DIR, "cistopic_obj_withModel.pkl"), "wb") as fh:
    pickle.dump(cistopic_obj, fh)


## 3. Clustering + UMAP

In [ ]:
find_clusters(
    cistopic_obj,
    target="cell",
    k=30,
    res=[1],
    prefix="pycisTopic_",
    scale=True,
    split_pattern="",
)

run_umap(cistopic_obj, target="cell", scale=True)

with open(os.path.join(Mallet_DIR, "cistopic_obj_withUMAP.pkl"), "wb") as fh:
    pickle.dump(cistopic_obj, fh)


## 4. Plot CellType UMAP -> PDF

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name="UMAP",
    variables=["celltype"],
    target="cell",
    num_columns=1,
    text_size=10,
    dot_size=1,
    color_dictionary={"celltype": celltype_colors},
)

fig = plt.gcf()
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("UMAP_1", fontsize=12, labelpad=10)
ax.set_ylabel("UMAP_2", fontsize=12, labelpad=10)
ax.set_title("")
ax.tick_params(axis="both", which="both", length=0)
ax.set_xticks([]); ax.set_yticks([])
for text in ax.texts:
    text.set_path_effects([withStroke(linewidth=0.7, foreground="black")])

plt.savefig(OUT_PDF, dpi=300, bbox_inches="tight", format="pdf")
plt.close()
print("UMAP PDF written to:", OUT_PDF)
